# **Laboratorio 8: Ready, Set, Deploy! 👩‍🚀👨‍🚀**

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Otoño 2026 </strong></center>

### Cuerpo Docente:

- Profesores: Pablo Badilla, Diego Cortez
- Auxiliares: Melanie Peña, Valentina Rojas
- Ayudantes: Javiera Arévalo, Tamara Carrasco y Ignacio Reyes

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Javiera Yañez Sanchez

### **Link de repositorio de GitHub:** https://github.com/javiyansan/MDS7202

## Temas a tratar

- Entrenamiento y registro de modelos usando MLFlow.
- Despliegue de modelo usando FastAPI
- Containerización del proyecto usando Docker

### Objetivos principales del laboratorio

- Generar una solución a un problema a partir de ML
- Desplegar su solución usando MLFlow, FastAPI y Docker

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

# **Introducción**

<p align="center">
  <img src="https://media.giphy.com/media/v1.Y2lkPTc5MGI3NjExODJnMHJzNzlkNmQweXoyY3ltbnZ2ZDlxY2c0aW5jcHNzeDNtOXBsdCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/AbPdhwsMgjMjax5reo/giphy.gif" width="400">
</p>



Consumida en la tristeza el despido de Renacín, Smapina ha decaído en su desempeño, lo que se ha traducido en un irregular tratamiento del agua. Esto ha implicado una baja en la calidad del agua, llegando a haber algunos puntos de la comuna en la que el vital elemento no es apto para el consumo humano. Es por esto que la sanitaria pública de la municipalidad de Maipú se ha contactado con ustedes para que le entreguen una urgente solución a este problema (a la vez que dejan a Smapina, al igual que Renacín, sin trabajo 😔).

El problema que la empresa le ha solicitado resolver es el de elaborar un sistema que les permita saber si el agua es potable o no. Para esto, la sanitaria les ha proveido una base de datos con la lectura de múltiples sensores IOT colocados en diversas cañerías, conductos y estanques. Estos sensores señalan nueve tipos de mediciones químicas y más una etiqueta elaborada en laboratorio que indica si el agua es potable o no el agua.

La idea final es que puedan, en el caso que el agua no sea potable, dar un aviso inmediato para corregir el problema. Tenga en cuenta que parte del equipo docente vive en Maipú y su intoxicación podría implicar graves problemas para el cierre del curso.

Atributos:

1. pH value
2. Hardness
3. Solids (Total dissolved solids - TDS)
4. Chloramines
5. Sulfate
6. Conductivity
7. Organic_carbon
8. Trihalomethanes
9. Turbidity

Variable a predecir:

10. Potability (1 si es potable, 0 no potable)

Descripción de cada atributo se pueden encontrar en el siguiente link: [dataset](https://www.kaggle.com/adityakadiwal/water-potability)

# **1. Optimización de modelos con Optuna + MLFlow (2.0 puntos)**

El objetivo de esta sección es que ustedes puedan combinar Optuna con MLFlow para poder realizar la optimización de los hiperparámetros de sus modelos.

Como aún no hemos hablado nada sobre `MLFlow` cabe preguntarse: **¡¿Qué !"#@ es `MLflow`?!**

<p align="center">
  <img src="https://media.tenor.com/eusgDKT4smQAAAAC/matthew-perry-chandler-bing.gif" width="400">
</p>

## **MLFlow**

`MLflow` es una plataforma de código abierto que simplifica la gestión y seguimiento de proyectos de aprendizaje automático. Con sus herramientas, los desarrolladores pueden organizar, rastrear y comparar experimentos, además de registrar modelos y controlar versiones.

<p align="center">
  <img src="https://spark.apache.org/images/mlflow-logo.png" width="350">
</p>

Si bien esta plataforma cuenta con un gran número de herramientas y funcionalidades, en este laboratorio trabajaremos con dos:
1. **Runs**: Registro que constituye la información guardada tras la ejecución de un entrenamiento. Cada `run` tiene su propio run_id, el cual sirve como identificador para el entrenamiento en sí mismo. Dentro de cada `run` podremos acceder a información como los hiperparámetros utilizados, las métricas obtenidas, las librerías requeridas y hasta nos permite descargar el modelo entrenado.
2. **Experiments**: Se utilizan para agrupar y organizar diferentes ejecuciones de modelos (`runs`). En ese sentido, un experimento puede agrupar 1 o más `runs`. De esta manera, es posible también registrar métricas, parámetros y archivos (artefactos) asociados a cada experimento.

### **Todo bien pero entonces, ¿cómo se usa en la práctica `MLflow`?**

Es sencillo! Considerando un problema de machine learning genérico, podemos registrar la información relevante del entrenamiento ejecutando `mlflow.autolog()` antes entrenar nuestro modelo. Veamos este bonito ejemplo facilitado por los mismos creadores de `MLflow`:

```python
#!pip install mlflow
import mlflow # importar mlflow

from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor

db = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(db.data, db.target)

# Create and train models.
rf = RandomForestRegressor(n_estimators=100, max_depth=6, max_features=3)

mlflow.autolog() # registrar automáticamente información del entrenamiento
with mlflow.start_run(): # delimita inicio y fin del run
    # aquí comienza el run
    rf.fit(X_train, y_train) # train the model
    predictions = rf.predict(X_test) # Use the model to make predictions on the test dataset.
    # aquí termina el run
```

Si ustedes ejecutan el código anterior en sus máquinas locales (desde un jupyter notebook por ejemplo) se darán cuenta que en su directorio *root* se ha creado la carpeta `mlruns`. Esta carpeta lleva el tracking de todos los entrenamientos ejecutados desde el directorio root (importante: si se cambian de directorio y vuelven a ejecutar el código anterior, se creará otra carpeta y no tendrán acceso al entrenamiento anterior). Para visualizar estos entrenamientos, `MLflow` nos facilita hermosa interfaz visual a la que podemos acceder ejecutando:

```
mlflow ui
```

y luego pinchando en la ruta http://127.0.0.1:5000 que nos retorna la terminal. Veamos en vivo algunas de sus funcionalidades!

<p align="center">
  <img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExZXVuM3A5MW1heDFpa21qbGlwN2pyc2VoNnZsMmRzODZxdnluemo2bCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/3o84sq21TxDH6PyYms/giphy.gif" width="400">
</p>

Les dejamos también algunos comandos útiles:

- `mlflow.create_experiment("nombre_experimento")`: Les permite crear un nuevo experimento para agrupar entrenamientos
- `mlflow.log_metric("nombre_métrica", métrica)`: Les permite registrar una métrica *custom* bajo el nombre de "nombre_métrica"


In [18]:
!uv add mlflow

  × No solution found when resolving dependencies for split (markers:               
  │ python_full_version >= '3.14' and sys_platform == 'win32'):
  ╰─▶ Because only the following versions of mlflow are available:
          mlflow<=2.13.0
          mlflow==2.13.1
          mlflow==2.13.2
          mlflow==2.14.0
          mlflow==2.14.1
          mlflow==2.14.2
          mlflow==2.14.3
          mlflow==2.15.0
          mlflow==2.15.1
          mlflow==2.16.0
          mlflow==2.16.1
          mlflow==2.16.2
          mlflow==2.17.0
          mlflow==2.17.1
          mlflow==2.17.2
          mlflow==2.18.0
          mlflow==2.19.0
          mlflow==2.20.0
          mlflow==2.20.1
          mlflow==2.20.2
          mlflow==2.20.3
          mlflow==2.20.4
          mlflow==2.21.0
          mlflow==2.21.1
          mlflow==2.21.2
          mlflow==2.21.3
          mlflow==2.22.0
          mlflow==2.22.1
          mlflow==2.22.2
          mlflow==2.22.3
          mlflow==2.22.4
         

Si tiene problemas puede necesitar ejecutar `uv add "setuptools<82.0.0"`

In [23]:
import mlflow  # importar mlflow
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

db = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(db.data, db.target)

# Create and train models.
rf = RandomForestRegressor(n_estimators=100, max_depth=6, max_features=3)

mlflow.autolog()  # registrar automáticamente información del entrenamiento
with mlflow.start_run():  # delimita inicio y fin del run
    # aquí comienza el run
    rf.fit(X_train, y_train)  # train the model
    predictions = rf.predict(X_test)  # Use the model to make predictions on the test dataset.
    # aquí termina el run

2026/06/10 10:54:17 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/06/10 10:54:17 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.
2026/06/10 10:54:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [24]:
run = mlflow.last_active_run()
info = mlflow.get_run(run.info.run_id)
print(info.data.params)
print(info.data.metrics)

{'bootstrap': 'True', 'ccp_alpha': '0.0', 'criterion': 'squared_error', 'max_depth': '6', 'max_features': '3', 'max_leaf_nodes': 'None', 'max_samples': 'None', 'min_impurity_decrease': '0.0', 'min_samples_leaf': '1', 'min_samples_split': '2', 'min_weight_fraction_leaf': '0.0', 'monotonic_cst': 'None', 'n_estimators': '100', 'n_jobs': 'None', 'oob_score': 'False', 'random_state': 'None', 'verbose': '0', 'warm_start': 'False'}
{'training_mean_squared_error': 1258.685737335556, 'training_mean_absolute_error': 29.482975693714803, 'training_r2_score': 0.7896954594818133, 'training_root_mean_squared_error': 35.47796129057525, 'training_score': 0.7896954594818133}


## **1.1 Combinando Optuna + MLflow (2.0 puntos)**

Ahora que tenemos conocimiento de ambas herramientas, intentemos ahora combinarlas para **más sabor**. El objetivo de este apartado es simple: automatizar la optimización de los parámetros de nuestros modelos usando `Optuna` y registrando de forma automática cada resultado en `MLFlow`.

Considerando el objetivo planteado, se le pide completar la función `optimize_model`, la cual debe:
- **Optimizar los hiperparámetros del modelo `XGBoost` usando `Optuna`.** Realice una cantidad de iteraciones para evitar tiempos de ejecución excesivos (al menos 10)
- **Registrar cada entrenamiento en un experimento nuevo**, asegurándose de que la métrica `f1-score` se registre como `"valid_f1"`. No se deben guardar todos los experimentos en *Default*; en su lugar, cada `experiment` y `run` deben tener nombres interpretables, reconocibles y diferentes a los nombres por defecto (por ejemplo, para un run: "XGBoost con lr 0.1").
- **Devolver el mejor modelo** usando la función `get_best_model` y serializarlo en el disco con `pickle.dump`. Luego, guardar el modelo en la carpeta `/models`.
- **Guardar el código en `optimize.py`**. La ejecución de `python optimize.py` debería ejecutar la función `optimize_model`.
- **Guardar las versiones de las librerías utilizadas** en el desarrollo.

*Hint: Le puede ser útil revisar los parámetros que recibe `mlflow.start_run`*

```python
def get_best_model(experiment_id):
    runs = mlflow.search_runs(experiment_id)
    best_model_id = runs.sort_values("metrics.valid_f1")["run_id"].iloc[0]
    best_model = mlflow.sklearn.load_model("runs:/" + best_model_id + "/model")

    return best_model
```

In [ ]:
%%writefile optimize.py
import os
import pickle
import mlflow
import mlflow.sklearn
import optuna
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

df = pd.read_csv("water_potability.csv")
df = df.fillna(df.median(numeric_only=True))  # imputar nulos con mediana

X = df.drop(columns=["Potability"])
y = df["Potability"]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

def get_best_model(experiment_id):
    runs = mlflow.search_runs(experiment_id)
    best_model_id = runs.sort_values("metrics.valid_f1", ascending=False)["run_id"].iloc[0]
    best_model = mlflow.sklearn.load_model("runs:/" + best_model_id + "/model")

    return best_model

def optimize_model():
    # Optimización de hiperparámetros del modelo con optuna y mlflow
    #-----------------------------------------------------------------
    # Nombre reconocible para experimento
    experiment = mlflow.get_experiment_by_name("Potabilidad_XGBoost_experimento")
    if experiment is None:
        experiment_id = mlflow.create_experiment("Potabilidad_XGBoost_experimento")
    else:
        experiment_id = experiment.experiment_id

    def objective_function(trial):
        # Definición de hiperparámetros
        params = {
            "n_estimators":  trial.suggest_int("n_estimators", 50, 400),
            "max_depth":     trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        }

        # Nombre interpretable para el run
        run_name = f"XGBoost con lr {params['learning_rate']:.3f} y depth {params['max_depth']}"

        # Entrenamiento de XGBoost (mlflow)
        with mlflow.start_run(experiment_id=experiment_id, run_name=run_name):
            model = XGBClassifier(seed=42, eval_metric="logloss", **params)
            model.fit(
                X_train, y_train, eval_set=[(X_train, y_train), (X_valid, y_valid)],
                )
            # Prediccion y evaluacion
            yhat = model.predict(X_valid)
            valid_f1 = f1_score(y_valid, yhat)

        # Registrar resultados en mlfloww
            mlflow.log_params(params)
            mlflow.log_metric("valid_f1", valid_f1)
            mlflow.sklearn.log_model(model, name="model")

        return valid_f1

    #-----------------------------------------------------------------
    
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_function, n_trials=15)

    # Obtener y guardar el mejor modelo
    best_model = get_best_model(experiment_id) # elige el con mayor f1
    os.makedirs("models", exist_ok=True) # crea carpeta models
    with open("models/best_model.pkl", "wb") as f:
        pickle.dump(best_model, f) # Guarda modelo

    print(f"Mejor F1: {study.best_value:.4f}")

    return best_model


# Guardar archivo optimize
if __name__ == "__main__":
    optimize_model()

Overwriting optimize.py


In [26]:
%run optimize.py

[I 2026-06-10 10:54:46,418] A new study created in memory with name: no-name-4f737d03-7a93-4995-a7c9-772fcc5cea5b


[0]	validation_0-logloss:0.63586	validation_1-logloss:0.64681
[1]	validation_0-logloss:0.60845	validation_1-logloss:0.63715
[2]	validation_0-logloss:0.58500	validation_1-logloss:0.63222
[3]	validation_0-logloss:0.55802	validation_1-logloss:0.62724
[4]	validation_0-logloss:0.53649	validation_1-logloss:0.62147
[5]	validation_0-logloss:0.52115	validation_1-logloss:0.62094
[6]	validation_0-logloss:0.50897	validation_1-logloss:0.61852
[7]	validation_0-logloss:0.50377	validation_1-logloss:0.61789
[8]	validation_0-logloss:0.48519	validation_1-logloss:0.61785
[9]	validation_0-logloss:0.47646	validation_1-logloss:0.61545
[10]	validation_0-logloss:0.46542	validation_1-logloss:0.61448
[11]	validation_0-logloss:0.45819	validation_1-logloss:0.61307
[12]	validation_0-logloss:0.45568	validation_1-logloss:0.61177
[13]	validation_0-logloss:0.44421	validation_1-logloss:0.60874
[14]	validation_0-logloss:0.42049	validation_1-logloss:0.60727
[15]	validation_0-logloss:0.41281	validation_1-logloss:0.60626
[1

2026/06/10 10:54:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/10 10:54:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-06-10 10:55:04,895] Trial 0 finished with value: 0.4597156398104265 and parameters: {'n_estimators': 138, 'max_depth': 8, 'learning_rate': 0.180381101867143}. Best is trial 0 with value: 0.4597156398104265.


[0]	validation_0-logloss:0.64180	validation_1-logloss:0.64895
[1]	validation_0-logloss:0.61940	validation_1-logloss:0.63722
[2]	validation_0-logloss:0.60071	validation_1-logloss:0.62992
[3]	validation_0-logloss:0.58755	validation_1-logloss:0.62429
[4]	validation_0-logloss:0.56395	validation_1-logloss:0.61968
[5]	validation_0-logloss:0.55144	validation_1-logloss:0.61752
[6]	validation_0-logloss:0.53646	validation_1-logloss:0.61126
[7]	validation_0-logloss:0.53113	validation_1-logloss:0.60839
[8]	validation_0-logloss:0.52243	validation_1-logloss:0.60785
[9]	validation_0-logloss:0.51607	validation_1-logloss:0.60618
[10]	validation_0-logloss:0.50433	validation_1-logloss:0.60710
[11]	validation_0-logloss:0.48487	validation_1-logloss:0.60210
[12]	validation_0-logloss:0.47932	validation_1-logloss:0.60097
[13]	validation_0-logloss:0.47204	validation_1-logloss:0.60128
[14]	validation_0-logloss:0.46987	validation_1-logloss:0.60025
[15]	validation_0-logloss:0.46777	validation_1-logloss:0.59970
[1

2026/06/10 10:55:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/10 10:55:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-06-10 10:55:22,832] Trial 1 finished with value: 0.4541284403669725 and parameters: {'n_estimators': 201, 'max_depth': 7, 'learning_rate': 0.1867987793874143}. Best is trial 0 with value: 0.4597156398104265.


...

[0]	validation_0-logloss:0.65579	validation_1-logloss:0.65584
[1]	validation_0-logloss:0.64295	validation_1-logloss:0.65081
[2]	validation_0-logloss:0.63113	validation_1-logloss:0.64711
[3]	validation_0-logloss:0.61924	validation_1-logloss:0.64159
[4]	validation_0-logloss:0.60736	validation_1-logloss:0.63898
[5]	validation_0-logloss:0.59719	validation_1-logloss:0.63751
[6]	validation_0-logloss:0.58641	validation_1-logloss:0.63415
[7]	validation_0-logloss:0.57759	validation_1-logloss:0.63284
[8]	validation_0-logloss:0.56881	validation_1-logloss:0.62988
[9]	validation_0-logloss:0.56071	validation_1-logloss:0.62692
[10]	validation_0-logloss:0.55380	validation_1-logloss:0.62534
[11]	validation_0-logloss:0.54711	validation_1-logloss:0.62371
[12]	validation_0-logloss:0.53723	validation_1-logloss:0.62242
[13]	validation_0-logloss:0.53066	validation_1-logloss:0.62141
[14]	validation_0-logloss:0.52455	validation_1-logloss:0.62017
[15]	validation_0-logloss:0.51777	validation_1-logloss:0.61958
[1

2026/06/10 10:58:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/10 10:58:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-06-10 10:58:37,872] Trial 13 finished with value: 0.4728132387706856 and parameters: {'n_estimators': 342, 'max_depth': 9, 'learning_rate': 0.06223882453686948}. Best is trial 3 with value: 0.49760765550239233.


[0]	validation_0-logloss:0.64491	validation_1-logloss:0.65054
[1]	validation_0-logloss:0.62190	validation_1-logloss:0.63646
[2]	validation_0-logloss:0.60906	validation_1-logloss:0.63114
[3]	validation_0-logloss:0.58342	validation_1-logloss:0.62330
[4]	validation_0-logloss:0.56528	validation_1-logloss:0.61875
[5]	validation_0-logloss:0.55782	validation_1-logloss:0.61692
[6]	validation_0-logloss:0.55389	validation_1-logloss:0.61448
[7]	validation_0-logloss:0.54394	validation_1-logloss:0.61105
[8]	validation_0-logloss:0.53126	validation_1-logloss:0.61208
[9]	validation_0-logloss:0.52727	validation_1-logloss:0.61247
[10]	validation_0-logloss:0.51242	validation_1-logloss:0.61008
[11]	validation_0-logloss:0.50557	validation_1-logloss:0.61184
[12]	validation_0-logloss:0.49825	validation_1-logloss:0.61241
[13]	validation_0-logloss:0.49471	validation_1-logloss:0.61382
[14]	validation_0-logloss:0.49133	validation_1-logloss:0.61213
[15]	validation_0-logloss:0.48722	validation_1-logloss:0.61206
[1

2026/06/10 10:58:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/10 10:58:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-06-10 10:58:54,379] Trial 14 finished with value: 0.4764044943820225 and parameters: {'n_estimators': 271, 'max_depth': 6, 'learning_rate': 0.22583579500291528}. Best is trial 3 with value: 0.49760765550239233.

Mejor F1: 0.4976


# **2. FastAPI (2.0 puntos)**

<div align="center">
  <img src="https://media3.giphy.com/media/YQitE4YNQNahy/giphy-downsized-large.gif" width="500">
</div>

Con el modelo ya entrenado, la idea de esta sección es generar una API REST a la cual se le pueda hacer *requests* para así interactuar con su modelo. En particular, se le pide:

- Guardar el código de esta sección en el archivo `main.py`. Note que ejecutar `python main.py` debería levantar el servidor en el puerto por defecto.
- Defina `GET` con ruta tipo *home* que describa brevemente su modelo, el problema que intenta resolver, su entrada y salida.
- Defina un `POST` a la ruta `/potabilidad/` donde utilice su mejor optimizado para predecir si una medición de agua es o no potable. Por ejemplo, una llamada de esta ruta con un *body*:

```json
{
   "ph":10.316400384553162,
   "Hardness":217.2668424334475,
   "Solids":10676.508475429378,
   "Chloramines":3.445514571005745,
   "Sulfate":397.7549459751925,
   "Conductivity":492.20647361771086,
   "Organic_carbon":12.812732207582542,
   "Trihalomethanes":72.28192021570328,
   "Turbidity":3.4073494284238364
}
```

Su servidor debería retornar una respuesta HTML con código 200 con:


```json
{
  "potabilidad": 0 # respuesta puede variar según el clasificador que entrenen
}
```

**`HINT:` Recuerde que puede utilizar [http://localhost:8000/docs](http://localhost:8000/docs) para hacer un `POST`.**

In [ ]:
%%writefile main.py
import pickle
import uvicorn
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel

# Cargar modelo
with open("models/best_model.pkl", "rb") as f:
    model = pickle.load(f)

# Definir estructura de entrada
class WaterMeasurement(BaseModel):
    ph: float
    Hardness: float
    Solids: float
    Chloramines: float
    Sulfate: float
    Conductivity: float
    Organic_carbon: float
    Trihalomethanes: float
    Turbidity: float

# Crear app
app = FastAPI()

# GET home
@app.get("/")
def home():
    return {
        "modelo": "XGBoost optimizado con Optuna",
        "problema": "Clasificación binaria de potabilidad del agua",
        "entrada": "9 mediciones químicas: ph, Hardness, Solids , Chloramines, Sulfate, Conductivity, Organic_carbon, Trihalomethanes, Turbidity",
        "salida": "potabilidad: 1 (potable) o 0 (no potable)"
    }

# POST predicción
@app.post("/potabilidad/")
def predecir_potabilidad(medicion: WaterMeasurement):
    datos = pd.DataFrame([medicion.model_dump()])
    prediccion = model.predict(datos)[0]
    return {"potabilidad": int(prediccion)}

if __name__ == "__main__":
    import nest_asyncio
    nest_asyncio.apply()
    uvicorn.run(app, host="0.0.0.0", port=8000)

Overwriting main.py


In [33]:
import subprocess
import sys

subprocess.Popen([sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"])

<Popen: returncode: None args: ['/Users/javierayanez/Documents/Semestre 11/M...>

INFO:     Started server process [16780]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:52530 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:52530 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:52533 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:52533 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:52538 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:52538 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:52540 - "POST /potabilidad/ HTTP/1.1" 200 OK
INFO:     127.0.0.1:52540 - "POST /potabilidad/ HTTP/1.1" 200 OK
INFO:     127.0.0.1:52552 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:52552 - "GET /openapi.json HTTP/1.1" 200 OK


# **3. Docker (2 puntos)**

<div align="center">
  <img src="https://miro.medium.com/v2/resize:fit:1400/1*9rafh2W0rbRJIKJzqYc8yA.gif" width="500">
</div>

Tras el éxito de su aplicación web para generar la salida, Smapina le solicita que genere un contenedor para poder ejecutarla en cualquier computador de la empresa de agua potable.

## **3.1 Creación de Container (1 punto)**

Cree un Dockerfile que use una imagen base de Python, copie los archivos del proyecto e instale las dependencias desde un `requirements.txt`. Con esto, construya y ejecute el contenedor Docker para la API configurada anteriormente. Entregue el código fuente (incluyendo `main.py`, `requirements.txt`, y `Dockerfile`) y la imagen Docker de la aplicación. Para la dockerización, asegúrese de cumplir con los siguientes puntos:

1. **Generar un archivo `.dockerignore`** que ignore carpetas y archivos innecesarios dentro del contenedor.
2. **Configurar un volumen** que permita la persistencia de los datos en una ruta local del computador.
3. **Exponer el puerto** para acceder a la ruta de la API sin tener que entrar al contenedor directamente.
4. **Incluir imágenes en el notebook** que muestren la ejecución del contenedor y los resultados obtenidos.
5. **Revisar y comentar los recursos utilizados por el contenedor**. Analice si los contenedores son livianos en términos de recursos.

## **3.2 Preguntas de Smapina (1 punto)**
Tras haber experimentado con Docker, Smapina desea profundizar más en el tema y decide realizarle las siguientes consultas:

- ¿Cómo se diferencia Docker de una máquina virtual (VM)?
- ¿Cuál es la diferencia entre usar Docker y ejecutar la aplicación directamente en el sistema local?
- ¿Cómo asegura Docker la consistencia entre diferentes entornos de desarrollo y producción?
- ¿Cómo se gestionan los volúmenes en Docker para la persistencia de datos?
- ¿Qué son Dockerfile y docker-compose.yml, y cuál es su propósito?

In [38]:
%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY main.py .
COPY models/ models/

EXPOSE 8000

CMD ["python", "main.py"]

Overwriting Dockerfile


In [41]:
%%writefile .dockerignore
.venv
__pycache__
*.pyc
.git
mlruns
mlflow.db
*.ipynb
.ipynb_checkpoints
optimize.py
water_potability.csv

Overwriting .dockerignore


In [50]:
!docker ps

CONTAINER ID   IMAGE                  COMMAND            CREATED         STATUS         PORTS                                         NAMES
bceebd90e3af   water-potability-api   "python main.py"   2 minutes ago   Up 2 minutes   0.0.0.0:8000->8000/tcp, [::]:8000->8000/tcp   water-api


In [53]:
!docker build -t water-potability-api .
!docker run -d -p 8000:8000 -v $(pwd)/models:/app/models --name water-api water-potability-api
!docker stats water-api --no-stream





[+] Building 0.0s (0/1)                                    docker:desktop-linux
[+] Building 0.2s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 228B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.11-slim        0.2s
[+] Building 0.3s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 228B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.11-slim        0.3s
[+] Building 0.5s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 228B                                       0.0s
 => [internal] load metadata for doc

![docker ps](ss2.png)

![docker stats](ss1.png)

![fastapi docs](ss3.png)

![post result](ss4.png)

Respuesta 3.1: 
> Según las capturas con los resultados de ejecución, el contenedor es bastante liviano en comparación con una VM.  Usa solo 138 MB de RAM para correr toda la API con el modelo XGBoost,  mientras que una VM requeriría varios GB solo para el sistema operativo.  El uso de CPU es prácticamente nulo (0.43%) en reposo, lo que confirma que  Docker es una solucion eficiente para desplegar aplicaciones como esta. Además, se configuró un volumen -v $(pwd)/models:/app/models para que el modelo persista localmente.

Respuesta 3.2:
> 1. Una maquina virtual VM virtualiza el hardware entero, es decir, instala un sistema operativo entero dentro de otro lo que puede ser muy pesado y lento. En cambio, Docker aisla la app a nivel proceso, no duplica el sistema operativo, sino que comparte el kernel de la maquina real, lo que lo hace mas liviano y rapido.
> 2. Hacerlo de manera local hace que dependa del entrono del computador, por lo que si algo se actualiza o cambia en la configuración, la app puede romperse. En cambio, Docker "empaqueta" la app con todas sus dependencias incluidas, lo que hace que pueda funcionar de forma aislada, permitiendo ser utilizado desde cualquier computador.
> 3. Docker mantiene la consistencia mediante el uso de imagenes (que no cambian), y como no cambia, a la hora de ejecución, será igual que cuando el programador lo ejecutó en su propia computadora (se utiliza una copia identica).
> 4. Los volumenes en Docker permiten que los datos persistan aunque el contenedor se detenga o elimine,  montando una carpeta del sistema local dentro del contenedor.
> 5. Dockerfile es un archivo de texto con las instrucciones para la construccion de la imagen , definiendo librerias, sistema base, etc.; y docker compose.yml es una archivo de configuracion para orquestar multiples contenedores a la vez, definiendo como se comunican y permitiendo encenderlo (ejemplo, comunicar la app del pc , una base de datos y mlflow).

# Conclusión

Éxito!
<div align="center">
  <img src="https://i.pinimg.com/originals/55/f5/fd/55f5fdc9455989f8caf7fca7f93bd96a.gif" width="500">
</div>